# 05r · 모듈 04-05 복습 (닫힌 책) (Review of Modules 04-05 (Closed Book))

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e Ch. 3 (§3.1, §3.3), Ch. 5 (§5.1–5.5) — 복습 범위 전체 · Ross 12e Ch. 7 (재생 보상, 나이/잔여, 월드 항등식) · Lawler 2e Ch. 5–6 · G&S 4e Ch. 10 (§10.1–10.4), Ch. 12 (§12.1–12.6) · Williams Ch. 10–11, §14.6 — 막히는 문항만 해당 절을 연다 |
| 선수 노트북 | 04a–04c (재생 과정), 05a–05d (마팅게일); 05e는 Q12에만 필요 (선택) |
| 예상 소요 | 60분 |
| 상태 | draft |

이 노트북에는 새 내용이 없다. 목적은 세 가지다.

1. 모듈 04–05의 핵심 정의·정리 12개를 **책 없이** 재구성하고 문항당 0–3점으로 자기 채점한다.
2. 세 개의 핵심 결과 — 재생-보상 정리(나이 과정과 $\pi(x)=1/E_x[T_x]$), 선택적 정지 정리(파산 문제), ABRACADABRA 논증(패턴 대기시간) — 를 **처음부터 재유도**하고, 코드로 정확값과 시뮬레이션을 모두 확인한다.
3. 자기채점표로 약한 노트북을 찾아 캡스톤 C3 전에 보강한다.

이 복습의 실은 하나다: **"오랜 시간 뒤에 어떻게 보이고, 그 사이 무엇이 보존되는가."** 모듈 04의 답은 *비율*이다 — 재생 구조가 있으면 장기 평균은 "사이클당 기대 / 사이클 길이 기대"로 붕괴한다. 모듈 05의 답은 *보존*이다 — 마팅게일은 고정 시각·정지시각·무한 시각에서 기댓값을 보존하고, 그 보존량 하나가 방정식 하나가 된다. 두 모듈은 04a의 기본 재생 정리 증명(월드 항등식)에서 만난다.

규칙: 1절과 3절의 ✍️ 칸은 책과 이전 노트북을 **닫고** 채운다. 정답(`<details>`)은 채운 뒤에만 연다. 코드 셀은 유도가 끝난 뒤 실행해 "내 손 계산이 맞았는지"를 기계로 확인하는 용도다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from scipy.integrate import quad
from scipy.stats import gamma as gamma_dist
from spkit.renewal import renewal_times, age_residual_process
from spkit.markov import (stationary, simulate_chain, empirical_distribution, tv_distance, absorption,
                          hitting_times, gamblers_ruin_matrix, gamblers_ruin_prob, gamblers_ruin_duration)

SEED, rng = rng_for("05r")
setup_plots()
N_REP = scale(400, minimum=40)    # R1: 재생 과정 복제 수 (FAST: 40)
N_STEPS = scale(200_000)          # R1b: 3상태 사슬 스텝 수 (FAST: 20_000)
N_PATHS = scale(20_000)           # R2: 파산 경로 수 (FAST: 2_000)
N_FLIP_PATHS = scale(100_000)     # R3: 동전 수열 수 (FAST: 10_000)
print(f"SEED={SEED}, fast={fast()}, N_REP={N_REP}, N_STEPS={N_STEPS}, N_PATHS={N_PATHS}, N_FLIP_PATHS={N_FLIP_PATHS}")

## 1. 닫힌 책 — 12문항 (Closed-Book Recall)

각 문항의 답을 종이(또는 아래 ✍️ 셀)에 먼저 적으세요. 한 문항당 2분을 넘기지 말고, 막히면 0점으로 두고 넘어가세요 — 어디가 약한지 아는 것이 이 절의 목적입니다. 다 적은 뒤 정답을 펼쳐 5절의 자기채점표에 점수(0–3)를 기록합니다. Q12는 05e의 내용이므로, 05e를 아직 풀지 않았다면 건너뛰고 총점에서 3점을 뺍니다.

**Q1.** (04a) 기본 재생 정리의 두 형태 $N(t)/t\to1/\mu$ a.s.와 $E[N(t)]/t\to1/\mu$ 중 두 번째가 첫 번째에서 바로 따라오지 않는 이유는? 증명에 쓰인 도구는?

<details><summary>정답</summary>

a.s. 수렴은 기댓값 수렴을 함의하지 않는다(균등적분가능성이 필요). 두 번째는 $N(t)+1$이 **정지시각**임을 이용한 월드 항등식 $E[S_{N(t)+1}]=\mu\,E[N(t)+1]$에서 출발한다 — $t<S_{N(t)+1}$이므로 하한 $E[N(t)]/t\ge1/\mu-1/t$을 얻고, 상한은 간격을 $M$에서 절단한 과정으로 얻는다. 즉 05c의 월드 항등식이 04a의 증명 도구였다.

</details>

**Q2.** (04a) 재생-보상 정리를 진술하고, 나이 교체 정책(수명 $F$, 교체 나이 $T$, 고장 비용 $c_f$, 예방 비용 $c_p$)의 장기 비용률을 써라.

<details><summary>정답</summary>

$R(t)/t\to E[R_1]/E[X_1]$ a.s. (사이클당 보상 기대 / 사이클 길이 기대; $E[X_1]<\infty$, $E|R_1|<\infty$). 비용률 $=\dfrac{c_fF(T)+c_p(1-F(T))}{\int_0^T(1-F(x))\,dx}$, 분모는 $E[\min(X,T)]$. 예: $X\sim U(0,1)$, $c_f=5$, $c_p=1$이면 비용률 $=\frac{4T+1}{T-T^2/2}$, 최소는 $T^*=0.5$에서 8 (교체 안 하면 10).

</details>

**Q3.** (04b) 평형(equilibrium) 분포 $F_e$의 밀도, 극한 평균 나이, 그리고 검사 역설의 정량 표현은?

<details><summary>정답</summary>

$f_e(x)=(1-F(x))/\mu$; 극한 평균 나이 = 극한 평균 잔여수명 $=E[X^2]/(2\mu)$; 시각 $t$를 덮는 간격(나이 + 잔여)의 극한 평균은 $E[X^2]/\mu\ge\mu$ (길이 편향, length bias; 등호는 상수 간격일 때만).

</details>

**Q4.** (04b) 푸아송 과정에서 극한 나이와 잔여수명이 모두 $\mathrm{Exp}(\lambda)$인 이유와 "버스 대기 $1/\lambda$" 역설의 해소는?

<details><summary>정답</summary>

지수분포는 $F_e=F$인 유일한 분포다(무기억성; $(1-F)/\mu=f$ ⇔ $f'=-\lambda f$). 잔여수명은 모든 $t$에서 정확히 $\mathrm{Exp}(\lambda)$이고, 나이는 $\min(t,\mathrm{Exp}(\lambda))$라 $t\to\infty$ 극한에서 $\mathrm{Exp}(\lambda)$. 도착 시각을 덮는 간격은 길이 편향되어 평균 $2/\lambda$이지만, 내가 기다리는 것은 그중 **잔여** 부분뿐이라 $1/\lambda$ — 두 값은 서로 다른 양이고 둘 다 옳다.

</details>

**Q5.** (04c) $\pi(x)=1/E_x[T_x]$의 증명 구조를 재생 언어로 말하라.

<details><summary>정답</summary>

$x$로의 연속 복귀 시각들은 강마르코프 성질로 **재생 과정**을 이룬다(i.i.d. 간격, 평균 $E_x[T_x]$). 사이클마다 "$x$에 머문 시간"을 보상 1로 두면 재생-보상 정리로 장기 점유 비율 $=1/E_x[T_x]$; 에르고딕 정리로 점유 비율 $=\pi(x)$. 둘을 등치하면 끝.

</details>

**Q6.** (05a) 마팅게일의 세 조건을 말하고, $S_n^2-n$ (SRW)이 마팅게일인 이유를 한 줄로 써라.

<details><summary>정답</summary>

적응(adapted), $E|M_n|<\infty$, $E[M_{n+1}\mid\mathcal F_n]=M_n$. $E[(S_n+X_{n+1})^2\mid\mathcal F_n]=S_n^2+2S_n\cdot0+1$ ⇒ $E[S_{n+1}^2-(n+1)\mid\mathcal F_n]=S_n^2-n$.

</details>

**Q7.** (05a) 폴리아 항아리의 빨간 비율이 마팅게일임을 계산으로 보여라.

<details><summary>정답</summary>

빨강 $R$, 전체 $N$개일 때 $E[R_{n+1}\mid\mathcal F_n]=R+R/N$ (확률 $R/N$로 빨강 하나 추가). 따라서 $E\!\left[\frac{R_{n+1}}{N+1}\,\middle|\,\mathcal F_n\right]=\frac{R(1+1/N)}{N+1}=\frac{R}{N}$.

</details>

**Q8.** (05b) 선택적 정지 정리(OST)의 세 충분조건과, 배가 전략(doubling)에서 어느 조건이 깨지는지 말하라.

<details><summary>정답</summary>

(a) $T$ 유계; (b) $T<\infty$ a.s. + 정지 과정 $M_{T\wedge n}$ 유계; (c) $E[T]<\infty$ + 증분 유계 $|M_{n+1}-M_n|\le c$. 배가 전략은 셋 다 깨진다: $T$ 무계, 정지 전 손실 $-(2^n-1)$ 무계, 증분 $2^n$ 무계. $E[T]=2<\infty$인데도 (c)가 안 되는 이유는 **증분** 조건 때문.

</details>

**Q9.** (05b) $p\ne1/2$ 파산 확률 공식과 그것을 주는 마팅게일은? 기대 시간은 어떤 마팅게일로?

<details><summary>정답</summary>

$P_i(S_T=N)=\dfrac{1-(q/p)^i}{1-(q/p)^N}$; $(q/p)^{S_n}$이 유계 마팅게일이므로 OST(b). 기대 시간은 $S_n-(p-q)n$과 OST(c)로: $E_i[T]=\dfrac{E_i[S_T]-i}{p-q}$.

</details>

**Q10.** (05c) 월드 항등식을 진술하고 $E[T_{HHH}]=14$를 겹침 논증으로 설명하라.

<details><summary>정답</summary>

$E[S_T]=E[X_1]\,E[T]$ ($T$ 정지시각, $E[T]<\infty$, $E|X_1|<\infty$). HHH: 매 시각 새 도박꾼이 1원을 걸고 공정 배당(2배)으로 패턴을 따라간다. 시각 $T$에 살아 있는 도박꾼은 겹침 길이 1, 2, 3에 대응하는 세 명이고 자본 $2+4+8=14$; 총 순손익 $\sum(\text{자본})-T$가 마팅게일이고 OST(c) ⇒ $E[T]=14$.

</details>

**Q11.** (05d) 마팅게일 수렴 정리의 조건과, 우도비 $L_n$이 0으로 가면서도 $E[L_n]=1$인 이유는?

<details><summary>정답</summary>

$\sup_nE|M_n|<\infty$ ($L^1$ 유계) ⇒ a.s. 수렴. $L_n=\prod_{i\le n}g(X_i)/f(X_i)$는 비음 마팅게일이라 수렴하고, SLLN으로 $\frac1n\log L_n\to-\mathrm{KL}(f\|g)<0$ ⇒ $L_n\to0$. 기댓값이 보존되지 않는 것은 **균등적분가능(UI)이 아니기** 때문 — 질량이 희귀한 큰 값으로 빠져나간다.

</details>

**Q12.** (05e; 아직 안 풀었으면 건너뛰기) 아주마 부등식을 진술하고 SRW $n=100$, $t=20$의 상한을 구하라. 둡 최대 부등식은?

<details><summary>정답</summary>

$|M_k-M_{k-1}|\le c_k$ ⇒ $P(M_n-M_0\ge t)\le\exp\!\left(-t^2/2\sum_kc_k^2\right)$; SRW $c_k=1$: $e^{-400/200}=e^{-2}\approx0.135$ (정확값 $P(S_{100}\ge20)=P(\mathrm{Bin}(100,\tfrac12)\ge60)\approx0.028$ — 부등식은 상한이지 근사가 아니다). 둡: 비음 서브마팅게일에 대해 $\lambda\,P(\max_{k\le n}X_k\ge\lambda)\le E[X_n]$.

</details>

✍️ 내 답 (책을 덮고 Q1–Q12의 답을 여기에 적은 뒤 정답을 펼치세요):

## 2. 복습 카드 (Review Cards) — 채점이 끝난 뒤에 펼치기

1절을 채점한 뒤에만 여세요. 3절의 재유도 중 막히면 여기로 돌아옵니다.

<details><summary>복습 카드 (모듈 04–05)</summary>

**재생 과정 (04a).** i.i.d. 양의 간격 $X_i$, $S_n=\sum_{i\le n}X_i$, $N(t)=\max\{n:S_n\le t\}$. 기본 재생 정리 $N(t)/t\to1/\mu$ a.s., $E[N(t)]/t\to1/\mu$. 재생-보상: $R(t)/t\to E[R_1]/E[X_1]$ a.s.

**나이·잔여 (04b).** $A(t)=t-S_{N(t)}$, $R(t)=S_{N(t)+1}-t$; 극한 분포 $F_e$, 밀도 $(1-F)/\mu$; 극한 평균 $E[X^2]/(2\mu)$; 덮는 간격 $A+R$의 극한 평균 $E[X^2]/\mu$.

**재생적 관점 (04c).** $\pi(x)=1/E_x[T_x]$; 에르고딕 정리 $\frac1n\sum_{k<n}f(X_k)\to\pi f$ (기약 + 양재귀; 비주기성 불필요).

**마팅게일 동물원 (05a).** 마팅게일/서브/슈퍼; $S_n$, $S_n^2-n$, $e^{\theta S_n}\cosh^{-n}\theta$, $Z_n/m^n$, 폴리아 비율, 우도비, 평균 1 곱; 둡 분해 $X_n=M_n+A_n$.

**정지시각·OST (05b).** $\{T\le n\}\in\mathcal F_n$; 정지 과정 $M_{T\wedge n}$도 마팅게일; OST (a) $T$ 유계 / (b) $T<\infty$ a.s. + 정지 과정 유계 / (c) $E[T]<\infty$ + 증분 유계 ⇒ $E[M_T]=M_0$; 파산 공식 4종($p=1/2$, $p\ne1/2$ 각각 확률·시간); 배가 전략과 $T_1=\inf\{n:S_n=1\}$ 반례($E[S_{T_1}]=1\ne0$).

**월드·패턴 (05c).** $E[S_T]=\mu E[T]$; 2차 $E[(S_T-\mu T)^2]=\sigma^2E[T]$; 패턴 대기시간 $E[T_A]=\sum_{k\in\mathrm{ov}(A)}2^k$ (공정 동전); KMP 자동자 + 흡수 사슬; Conway 공식(Penney).

**수렴 (05d).** $L^1$ 유계 ⇒ a.s. 수렴; $L^2$ 유계 ⇒ $L^2$ 수렴 + 기대 보존; UI ⇒ $E[M_\infty]=M_0$. 폴리아 $(a,b)$ 비율 → $\mathrm{Beta}(a,b)$; 분기 과정 $W=\lim Z_n/m^n$, $E[W]=1$ ($L^2$ 조건), $P(W=0)=q$.

**부등식 (05e).** 둡 최대 부등식, 콜모고로프, $L^2$ 부등식 $E[\max_{k\le n}M_k^2]\le4E[M_n^2]$; 아주마–회프딩; 둡 마팅게일 $E[f\mid\mathcal F_k]$와 유계 차분.

**정리 카드 (가정이 왜 필요한가).**
- *재생-보상 정리 (04a)*: $E[X_1]<\infty$, $E|R_1|<\infty$ ⇒ $R(t)/t\to E[R_1]/E[X_1]$. 두 기대의 유한성은 큰 수의 법칙을 사이클 수와 보상 합에 각각 적용하기 위해; 보상이 사이클 내부에서 임의로 누적되어도 사이클 경계 사이의 오차가 $o(t)$.
- *선택적 정지 정리 (05b)*: OST (a)/(b)/(c) 중 하나 ⇒ $E[M_T]=M_0$. 조건들은 모두 "극한과 기대의 교환"을 위한 것; 배가 전략·$T_1$이 조건 없이는 거짓임을 보이는 반례.
- *월드 항등식과 패턴 대기시간 (05c)*: $E[S_T]=\mu E[T]$; $E[T_A]=\sum_{k\in\mathrm{ov}(A)}2^k$. $E[T]<\infty$와 $T$가 정지시각(미래를 보지 않음)이어야 한다.
- *수렴 정리와 폴리아 극한 (05d)*: $L^1$ 유계 ⇒ a.s. 수렴; 폴리아 비율 → $\mathrm{Beta}(a,b)$. 같은 색 추가 규칙이 교환가능성(exchangeability)을 주어 극한 분포가 계산된다.

</details>

✍️ 내 말로: 재생-보상 비율(04)과 선택적 정지 보존(05)이 "같은 질문의 두 답"인 이유를, 04a의 $E[N(t)]/t$ 증명이 월드 항등식을 쓴다는 사실을 근거로 세 문장으로 써 보세요.

## 3. 재유도 — 세 가지 핵심 결과 (Re-derivations)

세 결과를 각각 **처음부터** 유도하세요. 각 결과마다 (i) 명제와 "가정이 왜 필요한가", (ii) ✍️ 내 유도 칸과 빈 코드 셀, (iii) 기대 답안(`<details>`), (iv) 정확값·시뮬레이션 코드 검사가 이어집니다. 코드는 유도가 끝난 뒤 실행합니다. 정확값은 코드가 공식에서 계산하므로, 코드 검사가 실패하면 시뮬레이션이 아니라 **내 손 계산**을 먼저 의심하세요.

### R1 재유도: 재생-보상으로 나이 과정과 $\pi(x)=1/E_x[T_x]$

**명제.** (i) 재생-보상 정리로 극한 평균 나이가 $E[X^2]/(2\mu)$임을 유도하라 (사이클 보상 $=\int_0^Xs\,ds=X^2/2$). (ii) 같은 방법으로 $\lim_tP(A(t)>1)=\int_1^\infty(1-F(x))\,dx/\mu$를 유도하고 $\mathrm{Gamma}(2,1)$ 간격에서 값을 구하라. 덮는 간격 $A+R$의 극한 평균도 구하라. (iii) $\pi(x)=1/E_x[T_x]$를 재생-보상으로 재유도하고, 3상태 사슬
$$P_3=\begin{pmatrix}0.5&0.5&0\\0.2&0.5&0.3\\0&0.4&0.6\end{pmatrix}$$
에서 $\pi$와 $E_0[T_0]$를 각각 계산해 곱이 1임을 보여라.

**가정이 왜 필요한가.**
- $E[X]<\infty$: 사이클 수 $N(t)/t\to1/\mu$에 SLLN을 쓰기 위해. $E[X^2]<\infty$: 사이클 보상 $X^2/2$의 기대가 유한해야 SLLN이 보상 합에 적용된다 (무거운 꼬리면 극한 평균 나이가 $\infty$).
- 보상이 사이클 안에서 연속적으로 쌓여도(나이의 적분) 마지막 미완 사이클의 기여는 $X_{N(t)+1}^2/2=o(t)$ a.s. — 이것이 "사이클 경계 사이 오차" 조건.
- (iii) 강마르코프 성질: 복귀 시각 사이의 조각들이 i.i.d.여야 복귀 시각들이 재생 과정을 이룬다. 기약 + 양재귀($E_x[T_x]<\infty$)가 그 재생 과정의 $\mu<\infty$를 보장한다.

✍️ 내 유도 (R1): (책을 덮고 여기에 (i)–(iii)의 유도와 숫자 답을 적으세요)

In [ ]:
# 여기에 풀이: 손으로 계산한 극한 평균 나이, P(A>1), 덮는 간격 평균, pi, E_0[T_0]를 적어 두고 아래 코드 검사와 비교

<details><summary>기대 답안 (R1)</summary>

**(i) 평균 나이.** 사이클 $k$의 길이 $X_k$ 동안 나이는 $0$에서 $X_k$까지 선형으로 자라므로 사이클 보상 $R_k=\int_0^{X_k}s\,ds=X_k^2/2$, i.i.d. 재생-보상 정리:
$$\frac1t\int_0^tA(s)\,ds\ \to\ \frac{E[X^2/2]}{E[X]}=\frac{E[X^2]}{2\mu}\quad\text{a.s.}$$
$\mathrm{Gamma}(2,1)$: $\mu=2$, $E[X^2]=\mathrm{Var}+\mu^2=2+4=6$ ⇒ $6/4=1.5$. (잔여수명은 사이클 안에서 $X$에서 0으로 줄어드는 대칭 그림이므로 같은 값.)

**(ii) 꼬리 확률.** 보상 $R_k=\int_0^{X_k}\mathbf 1_{\{s>1\}}ds=(X_k-1)^+$ ⇒ 장기 비율 $=E[(X-1)^+]/\mu=\int_1^\infty(1-F(x))\,dx/\mu$ (꼬리 적분 공식). $\mathrm{Gamma}(2,1)$의 생존함수 $1-F(x)=(1+x)e^{-x}$이므로 $\int_1^\infty(1+x)e^{-x}dx=\big[-(2+x)e^{-x}\big]_1^\infty=3/e$ ⇒ $P_e(A>1)=\dfrac{3}{2e}\approx0.5518$. (검산: 이것은 $\int_1^\infty f_e$, $f_e=(1-F)/\mu$와 같다.)
**덮는 간격.** 보상 $\int_0^{X_k}X_k\,ds=X_k^2$ ⇒ 극한 평균 $E[X^2]/\mu=3$ — 보통 간격 평균 2보다 길다(길이 편향).

**(iii) $\pi(x)=1/E_x[T_x]$.** $T_x^{(1)}<T_x^{(2)}<\cdots$를 $x$로의 연속 복귀 시각이라 하자. 강마르코프 성질로 사이클 $(X_{T^{(k)}},\dots,X_{T^{(k+1)}-1})$들은 i.i.d.이므로 복귀 시각들은 평균 $E_x[T_x]$인 재생 과정이다. 사이클당 보상 = "$x$에 머문 스텝 수" = 1 (사이클은 $x$에서 출발해 다음 복귀 직전까지이므로 $x$는 정확히 한 번). 재생-보상: $\frac1n\#\{k<n:X_k=x\}\to1/E_x[T_x]$. 에르고딕 정리(기약 + 양재귀)로 같은 좌변이 $\pi(x)$로 수렴 ⇒ $\pi(x)=1/E_x[T_x]$.

**$P_3$의 숫자.** $\pi P_3=\pi$: $\pi_0=0.5\pi_0+0.2\pi_1$ ⇒ $\pi_1=2.5\pi_0$; $\pi_2=0.4\pi_2+0.3\pi_1$ ⇒ $\pi_2=0.75\pi_1=1.875\pi_0$. 합 $=5.375\pi_0=1$ ⇒ $\pi=(8,20,15)/43\approx(0.1860,0.4651,0.3488)$.
$E_0[T_0]$: 상태 0을 흡수로 만들고 $h_i=E_i[\text{0 도달 시간}]$: $h_1=1+0.5h_1+0.3h_2$, $h_2=1+0.4h_1+0.6h_2$ ⇒ $h_2=2.5+h_1$, $0.5h_1=1+0.3(2.5+h_1)$ ⇒ $h_1=8.75$, $h_2=11.25$. $E_0[T_0]=1+0.5\cdot h_1=5.375=43/8$ ✓ $=1/\pi_0$. 같은 식으로 $E_1[T_1]=2.15=43/20$, $E_2[T_2]=2.8\overline6=43/15$.

</details>

코드 검사 1 (04a+04b) — 그림: $\mathrm{Gamma}(2,1)$ 재생 과정 한 경로의 나이 $A(t)$ 톱니 그림에 극한 평균 1.5를 겹친다.

In [ ]:
gamma21 = lambda n, g: g.gamma(2.0, 1.0, size=n)      # 간격 표본기: mu = 2, E[X^2] = 6

times_fig = renewal_times(gamma21, 210.0, rng)
t_fine = np.linspace(100, 200, 4001)
age_fig, _ = age_residual_process(times_fig, t_fine)

fig, ax = plt.subplots()
ax.plot(t_fine, age_fig, lw=1, label="age A(t)")
ax.axhline(1.5, color="C3", ls="--", lw=1, label="limiting mean E[X^2]/(2mu) = 1.5")
ax.set(xlabel="t", ylabel="A(t)", title="Age process of a Gamma(2,1) renewal process on [100, 200]")
ax.legend();

시뮬레이션: 복제 `N_REP`개의 독립 재생 과정을 $(0,1050]$ 위에 만들고, 격자 $t=100,101,\dots,1000$에서 (평균 나이, $A>1$ 비율, 평균 $A+R$)을 복제별로 잰다. 한 복제 안의 격자값들은 자기상관이 있지만 **복제 사이는 독립**이므로 복제 평균에 `mc_estimate`를 쓰는 것이 정당하다(03d의 배치 평균과 같은 논리). 정확값은 $\mathrm{Gamma}(2,1)$의 모멘트에서 코드가 계산한다.

In [ ]:
t_grid = np.linspace(100, 1000, 901)
mean_age_reps = np.empty(N_REP); frac_age_gt1_reps = np.empty(N_REP); mean_spread_reps = np.empty(N_REP)
for i in range(N_REP):
    times = renewal_times(gamma21, 1050.0, rng)
    age, res = age_residual_process(times, t_grid)
    assert np.isfinite(res).all()                 # 1000 이후에 재생이 있어야 잔여가 유한
    mean_age_reps[i] = age.mean()
    frac_age_gt1_reps[i] = (age > 1).mean()
    mean_spread_reps[i] = (age + res).mean()

MU, EX2 = 2.0 * 1.0, 2.0 * 1.0**2 + (2.0 * 1.0)**2       # Gamma(k=2, theta=1): mean k*theta, E[X^2] = Var + mean^2
mean_age_exact = EX2 / (2 * MU)                            # E[X^2] / (2 mu)
tail_int = quad(lambda x: gamma_dist.sf(x, 2.0, scale=1.0), 1.0, np.inf)[0]   # ∫_1^∞ (1-F(x)) dx (수치 적분)
p_age_gt1_exact = tail_int / MU                            # = E[(X-1)^+] / mu
np.testing.assert_allclose(p_age_gt1_exact, 3 / (2 * np.e), rtol=1e-8)      # 손 계산 3/(2e)와 대조
spread_exact = EX2 / MU                                    # E[X^2] / mu
r1_age, r1_frac, r1_spread = mc_estimate(mean_age_reps), mc_estimate(frac_age_gt1_reps), mc_estimate(mean_spread_reps)
print(f"mean age        : {r1_age}   exact {mean_age_exact:.6f}")
print(f"P(age > 1)      : {r1_frac}   exact {p_age_gt1_exact:.6f}")
print(f"mean A + R      : {r1_spread}   exact {spread_exact:.6f}  (ordinary gap mean {MU})")

코드 검사 2 (04c) — $P_3$에서 `stationary`로 $\pi$를 풀고, 한 긴 경로에서 0으로의 복귀 간격을 모아 $E_0[T_0]$를 추정한다. 복귀 간격은 강마르코프로 i.i.d.이므로 `mc_estimate`가 정당하다. 경로의 점유 비율과 $\pi$의 전변동 거리(TV distance)도 보고한다.

In [ ]:
P3 = np.array([[0.5, 0.5, 0.0], [0.2, 0.5, 0.3], [0.0, 0.4, 0.6]])
pi3 = stationary(P3)
pi3_hand = np.array([8, 20, 15]) / 43                                   # 기대 답안의 분수
np.testing.assert_allclose(pi3, pi3_hand, atol=1e-10)

# E_x[T_x]: 상태 x를 흡수로 둔 사슬의 기본 행렬 → h = N 1, E_x[T_x] = 1 + P[x, others] @ h
ET_return = np.empty(3)
for x in range(3):
    others = [s for s in range(3) if s != x]
    h = absorption(P3, others, [x])["t"]
    ET_return[x] = 1 + P3[x, others] @ h
print("pi            :", np.round(pi3, 6))
print("E_x[T_x]      :", np.round(ET_return, 6), "  pi * E_x[T_x] =", np.round(pi3 * ET_return, 10))
np.testing.assert_allclose(pi3 * ET_return, 1.0, atol=1e-10)

path = simulate_chain(P3, 0, N_STEPS, rng)[0]
visits0 = np.flatnonzero(path == 0)
r1b_gap = mc_estimate(np.diff(visits0))
emp3 = empirical_distribution(path[None, :], 3)
tv3 = tv_distance(emp3, pi3)
print(f"E_0[T_0]: MC {r1b_gap}   exact {ET_return[0]:.6f}")
print(f"occupation {np.round(emp3, 4)} vs pi {np.round(pi3, 4)}: TV = {tv3:.4f}")

### R2 재유도: 선택적 정지로 파산 확률과 기대 시간

**명제.** $\pm1$ 걸음, $P(+1)=p$, $q=1-p$, 상태 $\{0,\dots,N\}$, 출발 $i$, $T=\inf\{n:S_n\in\{0,N\}\}$. (i) $S_n$ ($p=1/2$), $S_n^2-n$ ($p=1/2$), $(q/p)^{S_n}$, $S_n-(p-q)n$이 마팅게일임을 보여라. (ii) OST를 각각에 적용해 $N=10$, $i=3$에서 $p=1/2$와 $p=0.45$의 탈출 확률 $P_i(S_T=N)$과 기대 시간 $E_i[T]$를 유도하라. 어떤 OST 조건을 어디에 썼는지 명시하라.

**가정이 왜 필요한가.**
- OST(b)를 $S_n$, $(q/p)^{S_n}$에 쓰려면 정지 과정이 유계여야 한다 — $0\le S_{T\wedge n}\le N$이라 성립. 상한 $N$이 없으면(무한 자본 상대) $T_1$ 반례처럼 $E[S_T]\ne S_0$.
- OST(c)를 $S_n^2-n$, $S_n-(p-q)n$에 쓰려면 $E[T]<\infty$와 유계 증분이 필요하다. $E[T]<\infty$는 "어느 위치에서든 $N$번 연속 $+1$이면 끝"에서 기하 꼬리로 얻는다. 증분 $|S_{n+1}^2-(n+1)-S_n^2+n|\le2N+2$ (정지 전에는 $|S_n|\le N$).
- $T<\infty$ a.s.만으로는 부족하다(함정): 시뮬레이션의 절단(`n_steps=500`)은 OST 위반을 감출 수 있으므로 절단된 경로가 없음을 코드에서 확인한다.

✍️ 내 유도 (R2): (책을 덮고 네 마팅게일의 증명과 네 숫자를 적으세요)

In [ ]:
# 여기에 풀이: 손으로 계산한 P_3(S_T=10), E_3[T] (p=1/2, p=0.45)를 적어 두고 아래 코드 검사와 비교

<details><summary>기대 답안 (R2)</summary>

**(i) 마팅게일.** $X$를 다음 걸음이라 하자($\mathcal F_n$과 독립).
- $E[S_{n+1}\mid\mathcal F_n]=S_n+E[X]=S_n+(p-q)$ ⇒ $S_n-(p-q)n$이 마팅게일; $p=1/2$면 $S_n$ 자체.
- $E[S_{n+1}^2\mid\mathcal F_n]=S_n^2+2S_nE[X]+1=S_n^2+1$ ($p=1/2$) ⇒ $S_n^2-n$.
- $E[(q/p)^{X}]=p\cdot\frac qp+q\cdot\frac pq=q+p=1$ ⇒ $E[(q/p)^{S_{n+1}}\mid\mathcal F_n]=(q/p)^{S_n}\cdot1$.
적분가능성은 각 $n$에서 $|S_n|\le n$으로 자명.

**(ii) $p=1/2$.** OST(b) on $S$: $E_3[S_T]=3$, $S_T\in\{0,10\}$ ⇒ $10\,P=3$ ⇒ $P_3(S_T=10)=0.3$. OST(c) on $S^2-n$: $E[S_T^2]-E[T]=9$, $E[S_T^2]=100\cdot0.3=30$ ⇒ $E_3[T]=21=i(N-i)$.

**(ii) $p=0.45$.** $r=q/p=0.55/0.45=11/9$. OST(b) on $r^{S}$: $r^3=P\,r^{10}+(1-P)$ ⇒ $P=\dfrac{1-r^3}{1-r^{10}}\approx0.128252$. OST(c) on $S-(p-q)n$: $E[S_T]-(p-q)E[T]=3$, $E[S_T]=10P$ ⇒ $E_3[T]=\dfrac{10P-3}{p-q}=\dfrac{10P-3}{-0.1}\approx17.1748$ (불리한 게임은 빨리 끝난다: 21 → 17.17).

</details>

코드 검사 (05b) — 정확값은 `gamblers_ruin_prob`/`gamblers_ruin_duration`(닫힌 형식)과 흡수 사슬의 기본 행렬(`absorption`) 두 갈래로 계산해 $10^{-10}$에서 맞춘 뒤, 경로 시뮬레이션과 비교한다. 절단된 경로(500스텝 안에 안 끝남)가 없음을 `assert`한다.

In [ ]:
N_RUIN, I0 = 10, 3
transient, absorbing = list(range(1, N_RUIN)), [0, N_RUIN]
ruin = {}
for p in (0.5, 0.45):
    P_ruin = gamblers_ruin_matrix(N_RUIN, p)
    ab = absorption(P_ruin, transient, absorbing)
    p_exact = gamblers_ruin_prob(I0, N_RUIN, p); t_exact = gamblers_ruin_duration(I0, N_RUIN, p)
    np.testing.assert_allclose([ab["B"][I0 - 1, 1], ab["t"][I0 - 1]], [p_exact, t_exact], rtol=1e-10)   # 두 갈래 일치

    paths = simulate_chain(P_ruin, I0, 500, rng, n_paths=N_PATHS)
    T = hitting_times(paths, {0, N_RUIN})
    assert np.isfinite(T).all(), "절단된 경로가 있음 — OST 조건 검사 불가"
    S_T = paths[np.arange(N_PATHS), T.astype(int)]
    ruin[p] = {"p_hat": mc_proportion(S_T == N_RUIN), "t_hat": mc_estimate(T), "p_exact": p_exact, "t_exact": t_exact}
    print(f"p={p}: P_3(S_T=10) MC {ruin[p]['p_hat']}  exact {p_exact:.6f} | E_3[T] MC {ruin[p]['t_hat']}  exact {t_exact:.6f}")

### R3 재유도: ABRACADABRA 논증과 폴리아 극한

**명제.** (i) 도박꾼 마팅게일 논증을 처음부터 써서 공정 동전 패턴 $A$의 첫 등장 시각이 $E[T_A]=\sum_{k\in\mathrm{ov}(A)}2^k$임을 얻고 (여기서 $\mathrm{ov}(A)=\{k:\ A\text{의 길이 }k\text{ 접두부}=\text{접미부}\}$), HTHT와 HHTT에 적용하라. (ii) 폴리아 항아리 $(1,2)$(빨강 1, 파랑 2)의 빨간 비율 극한 분포와 $P(X_\infty\le0.3)$을 구하고, 왜 극한이 상수가 아닌지 두 줄로 설명하라.

**가정이 왜 필요한가.**
- $E[T_A]<\infty$: 매 $m$번의 던지기 블록에서 확률 $2^{-m}$로 패턴이 나오므로 기하 꼬리 — OST(c)의 첫 조건. 증분은 한 시각에 살아 있는 도박꾼이 최대 $m$명, 자본 최대 $2^m$이라 유계.
- $T_A$는 정지시각이다(패턴 완성 여부는 지금까지의 동전으로 결정). "정확한 시각에 끝나는" 슬라이딩 윈도우 탐색이 아니면 $T$가 달라진다(05c 함정 5).
- (ii) 같은 색을 **추가**하는 규칙이 교환가능성을 주고, 그 덕에 $P(\text{앞 }n\text{번의 특정 색 순서})$가 순서에 무관해 $\mathrm{Beta}$ 극한이 계산된다.

✍️ 내 유도 (R3): (책을 덮고 도박꾼 논증과 두 패턴의 값, $\mathrm{Beta}$ 극한과 $P(X_\infty\le0.3)$을 적으세요)

In [ ]:
# 여기에 풀이: 손으로 계산한 E[T_HTHT], E[T_HHTT], P(X_inf <= 0.3)을 적어 두고 아래 코드 검사와 비교

<details><summary>기대 답안 (R3)</summary>

**(i) 도박꾼 논증.** 시각 $j=1,2,\dots$마다 새 도박꾼이 1원을 들고 와서 "$j$번째 동전이 $A_1$"에 걸고, 이기면 2원을 "$j+1$번째가 $A_2$"에 모두 걸고, … 패턴을 끝까지 맞추면 $2^m$원을 들고 떠난다(공정 배당이므로 각 판의 순손익은 평균 0). 도박꾼 전체의 총 순손익 $M_n=\sum_{j\le n}(\text{도박꾼 }j\text{의 자본}_n-1)$(카지노 순손익은 그 부호 반대)은 공정 게임의 합이라 마팅게일이고, 정지 전 증분 유계, $E[T_A]<\infty$ ⇒ OST(c): $E[M_{T_A}]=0$. 시각 $T_A$에 아직 살아 있는 도박꾼은 시각 $T_A-k+1$에 시작해 $k$번 연속 이긴 사람들, 즉 $A$의 접미부 $k$글자가 접두부 $k$글자와 같은 $k\in\mathrm{ov}(A)$ 각각에 한 명 — 자본 $2^k$. 따라서 $0=E[M_{T_A}]=\sum_{k\in\mathrm{ov}(A)}2^k-E[T_A]$.
- HTHT: 접두부/접미부 일치 길이 $k=2$ (HT), $k=4$ ⇒ $\mathrm{ov}=\{2,4\}$ ⇒ $E[T]=4+16=20$.
- HHTT: $k=4$만 ⇒ $E[T]=16$. (장기 빈도는 두 패턴 모두 $1/16$이지만 첫 등장 평균은 다르다 — 자기 겹침이 있는 패턴이 더 늦다.)

**KMP 재확인.** 상태 $s$ = 현재까지의 접미부 중 $A$의 접두부와 일치하는 최대 길이, $s=m$ 흡수. $E[T_A]=(N\mathbf 1)_0$, $N=(I-Q)^{-1}$ (01c) — 아래 코드에서 20, 16과 $10^{-9}$ 이내로 일치.

**(ii) 폴리아 $(1,2)$.** 빨간 비율 $X_n$은 $[0,1]$에 갇힌 유계 마팅게일 ⇒ a.s. 수렴(수렴 정리; UI이므로 $E[X_\infty]=X_0=1/3$). 교환가능성으로 앞 $n$번 중 빨강 $k$번인 특정 순서의 확률은 $\frac{\Gamma(a+b)}{\Gamma(a)\Gamma(b)}\frac{\Gamma(a+k)\Gamma(b+n-k)}{\Gamma(a+b+n)}$, 이는 $\mathrm{Beta}(a,b)$ 혼합 베르누이의 확률과 같다(de Finetti) ⇒ $X_\infty\sim\mathrm{Beta}(1,2)$, 밀도 $2(1-x)$, cdf $1-(1-x)^2$ ⇒ $P(X_\infty\le0.3)=1-0.49=0.51$.
**왜 상수가 아닌가.** 초반 몇 번의 뽑기가 항아리 구성을 영구히 바꾸므로 경로마다 다른 값으로 수렴한다("갇힘"). 보존되는 것은 **평균** $1/3$뿐이고 극한 자체는 확률변수다(함정: a.s. 수렴 ≠ 상수 극한).

</details>

코드 검사 (05c) — 05c와 같은 helper 두 개(슬라이딩 윈도우 첫 등장 시각, KMP 자동자 전이행렬)를 인라인으로 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의) — 05c와 동일
def first_occurrence(flips, pattern):
    """경로별로 pattern(0/1 배열, 길이 m)이 처음 완성되는 시각 T (슬라이딩 윈도우). 못 찾으면 n_flips + 1."""
    pattern = np.asarray(pattern, dtype=np.int8)
    n_paths, n_flips = flips.shape
    m = len(pattern)
    W = n_flips - m + 1
    match = np.ones((n_paths, W), dtype=bool)
    for j in range(m):
        match &= flips[:, j:W + j] == pattern[j]
    found = match.any(axis=1)
    T = match.argmax(axis=1) + m
    T[~found] = n_flips + 1
    return T

def kmp_matrix(pattern_str, p_head=0.5):
    """패턴 자동자 전이행렬 (m+1)x(m+1). 상태 s = 현재 일치 길이, s = m 흡수."""
    m = len(pattern_str)
    P = np.zeros((m + 1, m + 1))
    for s in range(m):
        for c, pc in (("H", p_head), ("T", 1 - p_head)):
            w = pattern_str[:s] + c
            nxt = max([L for L in range(1, min(len(w), m) + 1) if w[-L:] == pattern_str[:L]], default=0)
            P[s, nxt] += pc
    P[m, m] = 1.0
    return P

def to_bits(pattern_str):                                 # "HTHT" -> [1, 0, 1, 0]
    return np.array([1 if c == "H" else 0 for c in pattern_str], dtype=np.int8)

정확값 두 갈래: 겹침 공식 $\sum_{k\in\mathrm{ov}}2^k$와 KMP 흡수 사슬의 기대 흡수시간 — $10^{-9}$ 이내로 일치해야 한다. 그리고 `N_FLIP_PATHS`개의 길이 400 동전 수열에서 두 패턴의 첫 등장 시각을 잰다 ($P(T_{HTHT}>400)\approx0.9416^{400}\approx3.5\times10^{-11}$이라 절단은 사실상 없다; `assert`로 확인). $\mathrm{Beta}(1,2)$ cdf 값도 공식에서 계산한다.

In [ ]:
PATTERNS = ("HTHT", "HHTT")
exact_T = {}
for pat in PATTERNS:
    m = len(pat)
    ov = [k for k in range(1, m + 1) if pat[:k] == pat[m - k:]]
    by_overlap = float(sum(2.0**k for k in ov))
    by_chain = float(absorption(kmp_matrix(pat), list(range(m)), [m])["t"][0])
    assert abs(by_chain - by_overlap) < 1e-9
    exact_T[pat] = by_overlap
    print(f"{pat}: ov = {ov}, overlap formula {by_overlap:.1f}, KMP chain {by_chain:.10f}")

flips = rng.integers(0, 2, size=(N_FLIP_PATHS, 400), dtype=np.int8)
T_pat = {pat: first_occurrence(flips, to_bits(pat)) for pat in PATTERNS}
assert all((T_pat[pat] <= 400).all() for pat in PATTERNS), "절단된 경로가 있음"
T_HTHT, T_HHTT = T_pat["HTHT"], T_pat["HHTT"]
r3_htht, r3_hhtt = mc_estimate(T_HTHT), mc_estimate(T_HHTT)
print(f"E[T_HTHT]: MC {r3_htht}   exact {exact_T['HTHT']:.1f}   (sample sd {T_HTHT.std(ddof=1):.2f})")
print(f"E[T_HHTT]: MC {r3_hhtt}   exact {exact_T['HHTT']:.1f}   (sample sd {T_HHTT.std(ddof=1):.2f})")

polya_p_exact = 1 - (1 - 0.3)**2                          # Beta(1,2) cdf: 1 - (1-x)^2
print(f"Polya(1,2): P(X_inf <= 0.3) = {polya_p_exact:.4f},  E[X_inf] = a/(a+b) = {1 / (1 + 2):.4f}")

히스토그램: 두 패턴의 첫 등장 시각 분포. 평균 20 vs 16 — HTHT는 자기 겹침(HT) 때문에 "거의 됐다가 무너지는" 경로가 많아 꼬리가 길다.

In [ ]:
fig, ax = plt.subplots()
bins = np.arange(3.5, 120.5, 1.0)
for pat, color in zip(PATTERNS, ("C0", "C1")):
    ax.hist(T_pat[pat], bins=bins, density=True, alpha=0.5, color=color, label=f"T_{pat}")
    ax.axvline(exact_T[pat], color=color, ls="--", lw=1.2, label=f"exact mean E[T_{pat}] = {exact_T[pat]:.0f}")
ax.set(xlabel="first occurrence time T", ylabel="density", xlim=(0, 120),
       title=f"Waiting time for HTHT vs HHTT ({N_FLIP_PATHS} sequences)")
ax.legend();

## 4. Compare — 재유도 값 · 시뮬레이션 비교

정확값은 모두 위 코드가 공식(모멘트, 선형계, 기본 행렬, 겹침 집합)에서 계산한 것이다. "내 예측" 열에는 3절의 ✍️ 칸에 적은 **손 계산** 값을 넣어 보세요 (기본값 `None`).

In [ ]:
hand = {"R1_mean_age": None, "R1_p_age_gt1": None, "R1_cover": None, "R1b_E0T0": None,
        "R2_p_half": None, "R2_t_half": None, "R2_p_045": None, "R2_t_045": None,
        "R3_T_HTHT": None, "R3_T_HHTT": None}                       # 손 계산 값을 적어 넣으세요

rows = [
    {"name": "R1: long-run mean age, Gamma(2,1)",        "predicted": hand["R1_mean_age"],  "estimate": r1_age,             "exact": mean_age_exact},
    {"name": "R1: long-run P(age > 1)",                  "predicted": hand["R1_p_age_gt1"], "estimate": r1_frac,            "exact": p_age_gt1_exact},
    {"name": "R1: long-run mean covering interval A+R",  "predicted": hand["R1_cover"],     "estimate": r1_spread,          "exact": spread_exact},
    {"name": "R1b: E_0[T_0] for P3",                     "predicted": hand["R1b_E0T0"],     "estimate": r1b_gap,            "exact": ET_return[0]},
    {"name": "R2: P_3(S_T=10), p=1/2",                   "predicted": hand["R2_p_half"],    "estimate": ruin[0.5]["p_hat"],  "exact": ruin[0.5]["p_exact"]},
    {"name": "R2: E_3[T], p=1/2",                        "predicted": hand["R2_t_half"],    "estimate": ruin[0.5]["t_hat"],  "exact": ruin[0.5]["t_exact"]},
    {"name": "R2: P_3(S_T=10), p=0.45",                  "predicted": hand["R2_p_045"],     "estimate": ruin[0.45]["p_hat"], "exact": ruin[0.45]["p_exact"]},
    {"name": "R2: E_3[T], p=0.45",                       "predicted": hand["R2_t_045"],     "estimate": ruin[0.45]["t_hat"], "exact": ruin[0.45]["t_exact"]},
    {"name": "R3: E[T_HTHT]",                            "predicted": hand["R3_T_HTHT"],    "estimate": r3_htht,            "exact": exact_T["HTHT"]},
    {"name": "R3: E[T_HHTT]",                            "predicted": hand["R3_T_HHTT"],    "estimate": r3_hhtt,            "exact": exact_T["HHTT"]},
]
Markdown(compare_table(rows))

각 행이 어긋나면 무엇을 뜻하는가:

- **R1 mean age / P(age>1) / covering interval** — 어긋나면 $E[X^2]$를 $\mathrm{Var}$로 착각했거나(6 vs 2), 격자가 정상 영역에 못 미쳤거나(시작 100은 $\mu=2$의 50배라 충분), `age_residual_process`의 관례(첫 재생 전 $A=t$) 문제. 세 값 중 A+R만 어긋나면 잔여 쪽(1000 이후 재생 유무).
- **R1b E_0[T_0]** — 어긋나면 복귀 간격이 i.i.d.가 아니라는 뜻인데 강마르코프로 불가능하므로, `visits0` 인덱싱(첫 방문을 시각 0으로 셈)이나 $\pi$의 선형계 풀이를 의심. TV 거리가 크면 스텝 수 부족.
- **R2 (네 행)** — 확률이 어긋나면 $(q/p)^{S}$의 $r$을 $p/q$로 뒤집은 것; 시간이 어긋나면 $E[S_T]$에 확률 대신 $N$을 넣었거나 절단된 경로가 섞인 것(위 `assert`가 잡는다).
- **R3 (두 행)** — 어긋나면 `first_occurrence`가 슬라이딩 윈도우가 아니거나(05c 함정 5), 겹침 집합을 잘못 셌거나($\mathrm{ov}(HTHT)$에 $k=2$ 누락 → 16), 절단.

In [ ]:
assert_close(r1_age, mean_age_exact, k=4, name="R1 mean age")
assert_close(r1_frac, p_age_gt1_exact, k=4, name="R1 P(age>1)")
assert_close(r1_spread, spread_exact, k=4, name="R1 mean covering interval")
assert_close(r1b_gap, ET_return[0], k=4, name="R1b E_0[T_0]")
assert tv3 < (0.01 if not fast() else 0.03), f"TV distance {tv3:.4f} too large"
assert_close(ruin[0.5]["p_hat"], ruin[0.5]["p_exact"], k=4, name="R2 P, p=1/2")
assert_close(ruin[0.5]["t_hat"], ruin[0.5]["t_exact"], k=4, name="R2 E[T], p=1/2")
assert_close(ruin[0.45]["p_hat"], ruin[0.45]["p_exact"], k=4, name="R2 P, p=0.45")
assert_close(ruin[0.45]["t_hat"], ruin[0.45]["t_exact"], k=4, name="R2 E[T], p=0.45")
assert_close(r3_htht, exact_T["HTHT"], k=4, name="R3 E[T_HTHT]")
assert_close(r3_hhtt, exact_T["HHTT"], k=4, name="R3 E[T_HHTT]")
print("all MC-vs-exact checks passed (k = 4)")

## 5. 자기채점 (Self-Grading)

복습의 목적은 '오랜 시간 뒤 어떻게 보이고 무엇이 보존되는가'에 한 줄로 답할 수 있는지 확인하는 것이다: 재생 구조는 장기 평균을 사이클 비율로 붕괴시키고(04), 마팅게일은 고정·정지·무한 시각에서 기댓값을 보존하며(05), 둘은 월드 항등식에서 만난다. 아래 표를 채운 뒤 점수가 낮은 노트북부터 다시 연다.

**채점 규칙.** 3 = 책 없이 정확히(조건·상수까지), 2 = 핵심은 맞았으나 상수·조건 하나 누락, 1 = 방향만 맞음, 0 = 못 함. 오류 유형: **기억**(정의·공식을 잊음), **계산**(식은 알았으나 숫자·부호 실수), **개념**(왜 그런지 설명 불가). R1–R3는 유도의 완결성과 코드 검사 통과 여부로 채점한다.

| 문항 | 주제 (노트북) | 점수 (0–3) | 오류 유형 (기억/계산/개념) | 복습 노트북 |
|---|---|---|---|---|
| Q1 | 기본 재생 정리 두 형태·Wald (04a) | | | 04a 정리 1 |
| Q2 | 재생-보상·나이 교체 (04a) | | | 04a 정리 2 |
| Q3 | $F_e$·극한 나이·검사 역설 (04b) | | | 04b |
| Q4 | 푸아송의 나이/잔여·버스 역설 (04b) | | | 04b |
| Q5 | $\pi(x)=1/E_x[T_x]$ 재생 증명 (04c) | | | 04c |
| Q6 | 마팅게일 정의·$S_n^2-n$ (05a) | | | 05a |
| Q7 | 폴리아 비율 마팅게일 (05a) | | | 05a |
| Q8 | OST 세 조건·배가 전략 (05b) | | | 05b |
| Q9 | $p\ne1/2$ 파산 공식 (05b) | | | 05b |
| Q10 | 월드 항등식·HHH = 14 (05c) | | | 05c |
| Q11 | 수렴 정리·우도비 (05d) | | | 05d |
| Q12 | 아주마·둡 최대 부등식 (05e) | | | 05e |
| R1 | 재생-보상으로 나이·$\pi$ 재유도 | | | 04a / 04b / 04c |
| R2 | OST로 파산 확률·시간 재유도 | | | 05a / 05b |
| R3 | ABRACADABRA·폴리아 극한 재유도 | | | 05c / 05d |

**합격선.** 총점 45점 중 **36점 미만**이면 (Q12를 건너뛰었으면 42점 중 34점 미만) 표시된 노트북의 이론·Predict 셀을 다시 푼 뒤 C3로 진행한다. 같은 노트북에서 2점 이하가 둘 이상이면 그 노트북 전체를, "개념"이 붙은 문항은 해당 정리의 "가정이 왜 필요한가"를 다시 읽는다. `log/progress.md`에 한 줄을 남긴다.

**함정 카드 (traps) — 2점 이하를 준 문항이 아래 중 하나인지 확인하세요.**

- (04a) $N(t)/t\to1/\mu$ a.s.에서 $E[N(t)]/t\to1/\mu$가 자동으로 나오지 않는다 — 월드 항등식 + 정지시각 $N(t)+1$이 필요.
- (04b) 시각 $t$를 덮는 간격은 보통 간격보다 길다($E[X^2]/\mu$); 나이와 잔여의 합이지 '보통 간격'이 아니다.
- (04c) 에르고딕 정리(시간 평균)에는 비주기성이 필요 없지만 분포 수렴에는 필요하다.
- (05a) $E[M_n]=M_0$만으로는 마팅게일이 아니다; 조건부 기대가 필요.
- (05b) $T<\infty$ a.s. $\ne$ $E[T]<\infty$; 시뮬레이션의 절단은 OST 위반을 감춘다.
- (05c) 장기 빈도가 같아도 첫 등장 시각의 평균은 다르다(HHH 14 vs HTH 10; HTHT 20 vs HHTT 16); 기대 대기시간이 짧다고 먼저 나오는 것이 아니다(Penney 비추이성).
- (05d) a.s. 수렴 $\ne$ 기댓값 보존(우도비); 폴리아 극한은 상수가 아닌 $\mathrm{Beta}$ 확률변수.
- (05e) 아주마는 마팅게일 전용(드리프트를 먼저 제거); 부등식은 상한이지 근사가 아니다.

## 6. 정리

- 모듈 04: 재생 구조 ⇒ 장기 평균은 사이클 비율 $E[\text{보상}]/E[\text{길이}]$; 나이/잔여의 극한은 $F_e$, 덮는 간격은 길이 편향; $\pi(x)=1/E_x[T_x]$는 그 특수경우.
- 모듈 05: 마팅게일은 기댓값을 보존한다 — 고정 시각(정의), 정지시각(OST), 극한(UI). 보존량 하나가 방정식 하나(파산 확률, 대기시간).
- 두 모듈의 접점: 기본 재생 정리의 $E[N(t)]/t$ 증명은 정지시각 $N(t)+1$에 대한 월드 항등식이다.
- 계산 도구 세 벌을 일치시켰다: 마팅게일(보존량) = 흡수 사슬(기본 행렬) = 시뮬레이션(4 SE 이내).
- 자기채점 36/45 미만이면 표시된 노트북을 다시 풀고, 그 다음 C3(MCMC·이징: 01f/01e/05a 재사용)으로.

**Trap 카드.**
Q: 재생-보상 정리와 선택적 정지 정리는 서로 무관한 두 정리인가?
A: 아니다. 둘 다 "무작위 시각에서의 기대"를 다룬다. 기본 재생 정리 $E[N(t)]/t\to1/\mu$의 증명은 정지시각 $N(t)+1$에 대한 월드 항등식(OST의 따름정리)이고, 재생-보상 비율은 그 정리를 사이클 단위로 반복 적용한 것이다.

**다음 노트북: C3 (캡스톤 3 — MCMC와 이징 모형).** 01f의 메트로폴리스, 01e의 혼합, 05a의 마팅게일(디테일드 밸런스 검증에 조건부 기대)을 재사용한다. 자기채점표에서 점수가 낮은 노트북이 있으면 C3 전에 그 노트북을 먼저 다시 연다.

`log/progress.md` 한 줄 템플릿: `2026-__-__ | 05r | 총점 __/45 | 기억:__ 계산:__ 개념:__ | 다시 볼 노트북: __ | 메모: __`